In [107]:
import pandas as pd
from pathlib import Path

CLEANED_DF_PATH = Path("../data/processed/paysim_cleaned.parquet")
cleaned_df = pd.read_parquet(CLEANED_DF_PATH)

In [108]:
# see the cleaned df
cleaned_df.head()

,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFlaggedFraud,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,0,1
3,1,CASH OUT,181.000000,181.0,0.000000,21182.0,0.0,0,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0,0


In [109]:
# sender balance error
cleaned_df["senderBalanceError"] = (cleaned_df.oldBalanceOrig - cleaned_df.amount != cleaned_df.newBalanceOrig).astype("int8")

In [110]:
# receiver balance error
cleaned_df["receiverBalanceError"] = (cleaned_df.newBalanceDest - cleaned_df.oldBalanceDest != cleaned_df.amount).astype("int8")

In [111]:
# net changes
cleaned_df["netChangeOrig"] = cleaned_df.newBalanceOrig - cleaned_df.oldBalanceOrig
cleaned_df["netChangeDest"] = cleaned_df.newBalanceDest - cleaned_df.oldBalanceDest

In [112]:
# is the transaction large
cleaned_df["isLargeTransaction"] = (cleaned_df.amount > cleaned_df.amount.quantile(q=0.99)).astype("int8")

In [113]:
# time features
cleaned_df["day"] = (cleaned_df.step // 24).astype("int8")
cleaned_df["hour"] = (cleaned_df.step % 24).astype("int8")
cleaned_df["dayOfWeek"] = ((cleaned_df.day) % 7).astype("int8")

In [114]:
# amount to balance ratio
cleaned_df["amountToBalanceRatio"] = (cleaned_df.amount / (cleaned_df.oldBalanceOrig + 1)).astype("float32")

In [115]:
# move target to end
col_to_move = cleaned_df.pop("isFraud")
cleaned_df.insert(len(cleaned_df.columns), "isFraud", col_to_move)

In [116]:
# see final dataframe
cleaned_df.head()

,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFlaggedFraud,senderBalanceError,receiverBalanceError,netChangeOrig,netChangeDest,isLargeTransaction,day,hour,dayOfWeek,amountToBalanceRatio,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0,0,1,-9839.640625,0.0,0,0,1,0,0.057834,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0,0,1,-1864.279297,0.0,0,0,1,0,0.087731,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,0,0,1,-181.000000,0.0,0,0,1,0,0.994505,1
3,1,CASH OUT,181.000000,181.0,0.000000,21182.0,0.0,0,0,1,-181.000000,-21182.0,0,0,1,0,0.994505,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0,0,1,-11668.140625,0.0,0,0,1,0,0.280788,0


In [117]:
# save the feature engineered dataset
cleaned_df.to_parquet("../data/processed/paysim_feature_engineered.parquet", index=False)